In [13]:
# ========================================
# PRICE PREDICTION - DIRECTLY LOADING ALL EMBEDDINGS (5-FOLD CV)
# ========================================
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
import gc # Garbage Collector
import os

# ======================
# 1. DEVICE
# ======================
device = "cuda:1" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")


Using device: cuda:1


In [14]:
# ======================
# 2. LOAD ALL DATA
# ======================
print("Loading all image and text embeddings...")

# Load single files for all embeddings ---
all_img_emb = np.load("../embeddings/siglip2_train_image_embeddings.npy")
all_text_emb = np.load("../embeddings/all_text_emb.npy")
print(f"Image embeddings shape: {all_img_emb.shape}")
print(f"Text embeddings shape: {all_text_emb.shape}")

# --- Concatenate the full image and text embeddings to create the final feature matrix ---
X_all = np.concatenate([all_img_emb, all_text_emb], axis=1)

# --- Load and combine the dataframes to get the final target labels ---
all_df = pd.read_csv("../clean_data/train_cleaned.csv")
y_all = all_df["price"].values
# --- END MODIFIED SECTION ---

# --- Clean up original arrays to save memory ---
del all_img_emb, all_text_emb, all_df
gc.collect()
print(f"Final combined feature matrix shape: {X_all.shape}")
print(f"Final combined target array shape: {y_all.shape}")


# Optional: log-transform for long-tail prices
use_log = True
if use_log:
    print("Applying log1p transformation to target prices.")
    y_all = np.log1p(y_all)

Loading all image and text embeddings...
Image embeddings shape: (74999, 1536)
Text embeddings shape: (74999, 4096)
Final combined feature matrix shape: (74999, 5632)
Final combined target array shape: (74999,)
Applying log1p transformation to target prices.


In [15]:
# ======================
# 3. DATASET CLASS
# ======================
class PriceDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


In [16]:
# ======================
# 4. MODEL ARCHITECTURE
# ======================
class WideDeepMLP(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        # Deep layers
        self.deep1 = nn.Sequential(
            nn.Linear(input_dim, 1024),
            nn.BatchNorm1d(1024),
            nn.ReLU(),
            nn.Dropout(0.2)
        )
        self.deep2 = nn.Sequential(
            nn.Linear(1024, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.2)
        )
        self.deep3 = nn.Sequential(
            nn.Linear(512, 256),
            nn.ReLU()
        )

        # Residual projections
        self.res_proj1 = nn.Linear(1024, 512)
        self.res_proj2 = nn.Linear(512, 256)

        # Wide linear path
        self.wide = nn.Linear(input_dim, 64)

        # Attention over deep features
        self.attn = nn.Sequential(
            nn.Linear(256, 128),
            nn.Tanh(),
            nn.Linear(128, 1),
            nn.Sigmoid()
        )

        # Final output
        self.out = nn.Linear(256 + 64, 1)
        self.softplus = nn.Softplus() # ensure positive predictions

    def forward(self, x):
        d1 = self.deep1(x)                             # 1024
        d2 = self.deep2(d1) + self.res_proj1(d1)      # 512
        d3 = self.deep3(d2) + self.res_proj2(d2)      # 256

        attn_weights = self.attn(d3)
        d3 = d3 * attn_weights                       # apply attention

        wide_feat = self.wide(x)                     # 64
        combined = torch.cat([d3, wide_feat], dim=1)

        return self.softplus(self.out(combined)).squeeze(1)


In [17]:
# ======================
# 5. SMAPE METRIC
# ======================
def smape(y_true, y_pred):
    # If log-transformed, revert back for metric calculation
    if use_log:
        y_true = torch.expm1(y_true)
        y_pred = torch.expm1(y_pred)
    numerator = torch.abs(y_pred - y_true)
    denominator = (torch.abs(y_true) + torch.abs(y_pred)) / 2 + 1e-8
    return 100 * torch.mean(numerator / denominator)


In [24]:

# ======================
# 6. CROSS-VALIDATION & TRAINING
# ======================
N_SPLITS = 5
EPOCHS = 100
BATCH_SIZE = 128
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 1e-5

kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=42)

for fold, (train_idx, val_idx) in enumerate(kf.split(X_all)):
    print("-" * 50)
    print(f"🚀 STARTING FOLD {fold+1}/{N_SPLITS}")
    print("-" * 50)
    
    # --- Data for current fold ---
    X_train, X_val = X_all[train_idx], X_all[val_idx]
    y_train, y_val = y_all[train_idx], y_all[val_idx]
    
    train_ds = PriceDataset(X_train, y_train)
    val_ds = PriceDataset(X_val, y_val)
    
    train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
    val_dl = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
    
    # --- Initialize model and optimizer for current fold ---
    input_dim = X_train.shape[1]
    model = WideDeepMLP(input_dim=input_dim).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
    loss_fn = nn.L1Loss() # MAE Loss is suitable for price prediction
    
    best_val_smape = float("inf")
    
    # --- Training loop for current fold ---
    for epoch in range(EPOCHS):
        model.train()
        total_train_loss = 0
        for X_batch, y_batch in train_dl:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            
            optimizer.zero_grad()
            preds = model(X_batch)
            loss = loss_fn(preds, y_batch)
            loss.backward()
            optimizer.step()
            
            total_train_loss += loss.item() * len(y_batch)
        
        train_mae = total_train_loss / len(train_ds)

        # --- Validation for current fold ---
        model.eval()
        all_preds = []
        all_targets = []
        with torch.no_grad():
            for X_batch, y_batch in val_dl:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)
                preds = model(X_batch)
                all_preds.append(preds)
                all_targets.append(y_batch)
        
        all_preds_tensor = torch.cat(all_preds)
        all_targets_tensor = torch.cat(all_targets)
        val_smape = smape(all_targets_tensor, all_preds_tensor).item()
        
        scheduler.step()
        
        print(f"Fold {fold+1}, Epoch {epoch+1:02d}: Train MAE={train_mae:.4f}, Val SMAPE={val_smape:.4f}")
        
        # --- Save best model for current fold ---
        if val_smape < best_val_smape:
            best_val_smape = val_smape
            model_save_path = f"best_price_model_fold_{fold}.pt"
            torch.save(model.state_dict(), model_save_path)
            print(f"✅ Saved best model for fold {fold+1} to {model_save_path} (SMAPE: {best_val_smape:.4f})")

print("\n🎉 Cross-validation training complete.")

--------------------------------------------------
🚀 STARTING FOLD 1/5
--------------------------------------------------


RuntimeError: Error(s) in loading state_dict for WideDeepMLP:
	Missing key(s) in state_dict: "deep1.0.weight", "deep1.0.bias", "deep1.1.weight", "deep1.1.bias", "deep1.1.running_mean", "deep1.1.running_var", "deep2.0.weight", "deep2.0.bias", "deep2.1.weight", "deep2.1.bias", "deep2.1.running_mean", "deep2.1.running_var", "deep3.0.weight", "deep3.0.bias", "res_proj1.weight", "res_proj1.bias", "res_proj2.weight", "res_proj2.bias", "attn.0.weight", "attn.0.bias", "attn.2.weight", "attn.2.bias". 
	Unexpected key(s) in state_dict: "image_encoder.0.weight", "image_encoder.0.bias", "image_encoder.1.weight", "image_encoder.1.bias", "image_encoder.1.running_mean", "image_encoder.1.running_var", "image_encoder.1.num_batches_tracked", "image_encoder.4.weight", "image_encoder.4.bias", "image_encoder.5.weight", "image_encoder.5.bias", "image_encoder.5.running_mean", "image_encoder.5.running_var", "image_encoder.5.num_batches_tracked", "image_encoder.8.weight", "image_encoder.8.bias", "text_encoder.

In [28]:
# Load single files for all embeddings ---
import numpy as np
all_img_emb = np.load("../embeddings/updated_siglip2_test_image_embeddings_no_zero.npy")
all_text_emb = np.load("../embeddings/mistral_test_embeddings_from_notebook.npy")

# --- Concatenate the full image and text embeddings to create the final feature matrix ---
X_test = np.concatenate([all_img_emb, all_text_emb], axis=1)

In [ ]:
input_dim = X_test.shape[1]
model0 = WideDeepMLP(input_dim=input_dim).to(device)
model0.load_state_dict(torch.load("./best_price_model_fold_0.pt", map_location=device))
model0.eval()
model1 = WideDeepMLP(input_dim=input_dim).to(device)
model1.load_state_dict(torch.load("./best_price_model_fold_1.pt", map_location=device))
model1.eval()
model2 = WideDeepMLP(input_dim=input_dim).to(device)
model2.load_state_dict(torch.load("./best_price_model_fold_2.pt", map_location=device))
model2.eval()
model3 = WideDeepMLP(input_dim=input_dim).to(device)
model3.load_state_dict(torch.load("./best_price_model_fold_3.pt", map_location=device))
model3.eval()
model4 = WideDeepMLP(input_dim=input_dim).to(device)
model4.load_state_dict(torch.load("./best_price_model_fold_4.pt", map_location=device))
model4.eval()

RuntimeError: Error(s) in loading state_dict for WideDeepMLP:
	size mismatch for deep1.0.weight: copying a param with shape torch.Size([1024, 1536]) from checkpoint, the shape in current model is torch.Size([1024, 5632]).
	size mismatch for wide.weight: copying a param with shape torch.Size([64, 1536]) from checkpoint, the shape in current model is torch.Size([64, 5632]).

: 

In [10]:
all_preds = []

with torch.no_grad():
    for X_batch in test_dl:
        X_batch = X_batch.to(device)
        preds = model0(X_batch)
        all_preds.append(preds.cpu())

y_pred_test1 = np.expm1(torch.cat(all_preds).numpy())

all_preds = []

with torch.no_grad():
    for X_batch in test_dl:
        X_batch = X_batch.to(device)
        preds = model1(X_batch)
        all_preds.append(preds.cpu())

y_pred_test2 = np.expm1(torch.cat(all_preds).numpy())

all_preds = []

with torch.no_grad():
    for X_batch in test_dl:
        X_batch = X_batch.to(device)
        preds = model2(X_batch)
        all_preds.append(preds.cpu())

y_pred_test3 = np.expm1(torch.cat(all_preds).numpy())

all_preds = []

with torch.no_grad():
    for X_batch in test_dl:
        X_batch = X_batch.to(device)
        preds = model3(X_batch)
        all_preds.append(preds.cpu())

y_pred_test4 = np.expm1(torch.cat(all_preds).numpy())

all_preds = []

with torch.no_grad():
    for X_batch in test_dl:
        X_batch = X_batch.to(device)
        preds = model4(X_batch)
        all_preds.append(preds.cpu())

y_pred_test5 = np.expm1(torch.cat(all_preds).numpy())

y_pred_test = (y_pred_test1 + y_pred_test2 + y_pred_test3 + y_pred_test4 + y_pred_test5)/5

In [ ]:
import pandas as pd

test_df = pd.read_csv("../dataset/test.csv")
submission_df = pd.DataFrame({
    "sample_id": test_df["sample_id"],
    "price": y_pred_test
})

submission_df.to_csv("submission.csv", index=False)
print("💾 Predictions saved to submission.csv")

💾 Predictions saved to submission.csv


: 